# 여자어 번역기


1. 사용자 입력을 음성으로 받는다.
2. STT를 적용하여 텍스트로 변환한다.
3. 변환된 텍스트를 입력으로 하여 프롬프트 엔지니어링을 해 api 요청을 보낸다.
4. 반환받은 응답을 TTS를 적용하여 음성으로 재생한다.
 (+) 2에서 입력된 텍스트와 4에서 반환된 응답을 채팅 내역 보듯이 (카카오톡 대화처럼) 현출되도록 출력한다.

In [37]:
from dotenv import load_dotenv
load_dotenv()

True

### [편의를 위한 Text 입력후 mp3파일로 저장]

In [38]:
import streamlit as st
from openai import OpenAI
import speech_recognition as sr
import tempfile, os, json
from datetime import datetime
 


In [39]:
client = OpenAI()
input_text = '기분 좋아보인다?'

with client.audio.speech.with_streaming_response.create(
    model='gpt-4o-mini-tts',
    voice='shimmer',
    input=input_text
) as response:
    response.stream_to_file(f'./tts/{input_text}.mp3')

In [40]:
from IPython.display import Audio

display(Audio(f'./tts/{input_text}.mp3', autoplay=True))

### 입력

In [41]:
with open(f'./tts/{input_text}.mp3', 'rb') as f:
    user_text = client.audio.transcriptions.create(
        model='whisper-1',
        file=f
    )

print(user_text.text)

기분 좋아 보인다.


### 상황 부여

In [42]:
situation_ex = {
    "남자친구와 싸운 후": "남자친구와 방금 싸운 직후의 냉랭한 상황",
    "남자친구와 평상시 대화": "남자친구와 일상적으로 대화하는 평화로운 상황",
    "친구들과 노는 중": "친한 친구들과 함께 신나게 놀고 있는 상황",
    "가족과 식사 중": "가족과 함께 식사하며 대화하는 상황",
    "직장/학교에서 스트레스받은 후": "직장이나 학교에서 힘든 일이 있었던 상황",
    "기분 좋고 설레는 날": "무언가 좋은 일이 생겨 기분이 들뜬 상황",
    "지루하고 무료한 날": "딱히 할 일 없이 무기력하고 심심한 상황",
}
 
target_ex = {
    "남자친구": "남자친구에게 하는 말",
    "남사친 (남자사람친구)": "남자사람친구에게 하는 말",
    "여자친구 (친구)": "여자친구(동성 친한 친구)에게 하는 말",
    "가족": "가족에게 하는 말",
    "직장 동료 / 선배": "직장 동료나 선배에게 하는 말",
    "단톡방 (여러 명)": "단체 카카오톡 채팅방에서 여러 명에게 하는 말",
}

### 번역부분 (프롬프트 엔지니어링)

In [43]:
import json


def translate(user_text: str, situation: str, target: str) -> dict:
 
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        max_tokens=1000,
        messages=[
            {
                "role": "system",
                "content": f"""
                당신은 '여자어 번역기'입니다. 여자가 하는 말의 진짜 숨겨진 의도를 해석해주세요.
 
                [현재 상황]: {situation_ex[situation]}
                [말하는 대상]: {target_ex[target]}

                위 상황과 대상을 반드시 반영하여 해석하세요. 같은 말이라도 상황과 대상에 따라 전혀 다른 의미를 가집니다.

                반드시 아래 JSON 형식으로만 답하세요. 코드블록 없이 순수 JSON만:
                {{
                  "translation": "진짜 의미 설명 (2~3문장, 상황과 대상을 반영한 구체적 해석)",
                  "points": ["포인트1", "포인트2", "포인트3"]
                }}
                """
            },
            {
                "role": "user",
                "content": f'다음 말을 여자어로 번역해줘: "{user_text}"'
            }
        ]
    )
    raw = response.choices[0].message.content.strip().replace("```json", "").replace("```", "")
    return json.loads(raw)